# Day 17 — Pandas intro: Series, DataFrame, indexing, selection
Objectives:
- Create Series and DataFrames.
- Index/column selection, boolean masks, loc/iloc.
- Basic inspection and summary.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-17`. Read
`python/ds-60day/companion-guides/day17_pandas_intro.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A Series is a one-dimensional labeled array; a DataFrame is a table of
labeled columns sharing an index. Before manipulating a DataFrame,
state its **row grain**—what one row represents—and what makes a row
unique. Column names carry variable meaning; the index carries row
labels, which may or may not be a business key.

`loc` selects by labels and boolean masks; `iloc` selects by integer
positions. Pandas aligns many operations by index labels rather than
blindly by current row position. Prefer vectorized column operations
and explicit `.loc` assignment. Inspect shape, dtypes, missingness, and
a small sample before trusting a calculation.

### Vocabulary

- **Series:** a one-dimensional array with an index and optional name.
- **DataFrame:** a two-dimensional collection of aligned labeled columns.
- **index:** the labels identifying rows for selection and alignment.
- **row grain:** the real-world meaning of one row.
- **boolean mask:** a True/False Series used to select matching rows.
- **vectorized operation:** a column/array operation applied without an explicit Python row loop.

## Syntax anatomy

`frame.loc[mask, ["name", "score"]]` has two selectors: rows before the
comma and columns after it. `frame.iloc[:3, 0]` uses positions instead
of labels. `frame["rate"] = frame["part"] / frame["whole"]` aligns the
two Series by index and assigns the computed Series under a new column
label.

### Worked example 1 — Build and inspect a tiny labeled table

Use constructed data so the example is fully offline. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import pandas as pd

sales = pd.DataFrame({
    "order_id": [101, 102, 103],
    "region": ["west", "east", "west"],
    "amount": [20.0, 35.0, 15.0],
})
(sales.shape, sales.dtypes.astype(str).to_dict(), sales["order_id"].is_unique)

**Expected observation:** `((3, 3), {...}, True)`. The dtype spellings may vary slightly; each row represents one order and `order_id` is unique here.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Filter rows and derive a column without a loop

A mask selects west orders; a vectorized expression creates tax. Predict first; then run the next cell.

In [ ]:
west = sales.loc[sales["region"].eq("west"), ["order_id", "amount"]]
sales = sales.assign(amount_with_tax=sales["amount"] * 1.08)
(west["order_id"].tolist(), sales["amount_with_tax"].round(2).tolist())

**Expected observation:** `([101, 103], [21.6, 37.8, 16.2])`. The original row index is retained in `west`.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Print shape, column names, dtypes, and row grain before writing a selection.
2. Use `.loc` for labels/masks and `.iloc` for positions; never infer from integer-looking labels.
3. Check mask index alignment when a correct-looking mask selects the wrong rows or raises.
4. Use `.copy()` for an intentionally independent filtered frame and `.loc` for explicit assignment.

**Alternative to compare:** Method chaining can express a readable pipeline; named intermediate frames are better while learning or debugging each contract.

**Boundary to test:** Empty selections, duplicate indexes, zero denominators, missing values, and chained assignment need explicit behavior.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
import pandas as pd
import seaborn as sns
df = sns.load_dataset('tips')
df.head(), df.info(), df.describe(include='all')


In [ ]:
# Selection
df['total_bill'].head()
df[['total_bill','tip']].head()
df.loc[df['day']=='Sun', ['total_bill','tip','size']].head()
df.iloc[:5, :3]


## Indexing
Set and reset index; sort_index.

In [ ]:
df2 = df.set_index(['day','time']).sort_index()
df2.loc[('Sun','Dinner')].head()
df2.reset_index().head()


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Using the lesson DataFrame, select rows where `time == 'Dinner'`, then compute the mean of the selected `tip` values. **Before code:** state the row grain and write the mask separately.
   **Expected behavior:** the result is one scalar equal to `dinner_rows['tip'].mean()`. **Constraints:** use `.loc` and do not loop.
   **Verify:** assert every selected row is Dinner and the selection is non-empty before reporting the mean.

2. Add a Boolean `is_big_party` column that is `True` exactly when `size >= 5`. **Constraints:** use one vectorized comparison and explicit assignment; do not use row-wise `apply`.
   **Verify:** compare the new column with `df['size'].ge(5)`, inspect both True and False rows, and confirm row count/index are unchanged.

3. Create a safe `tip_rate = tip / total_bill`, treating a zero bill as missing rather than infinity, then sort descending by `tip_rate`. **Constraints:** preserve the unsorted source in a separate name and choose where missing rates appear.
   **Verify:** assert there are no infinite values and that non-missing rates are monotonically decreasing.

### Additional mastery practice

State a DataFrame's row grain, column meanings, and index role before selecting or deriving data. Prefer vectorized, explicit assignments.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

4. **Prediction:** Predict the difference between `.loc[labels]` and `.iloc[positions]` after an integer index has been reordered.
   **Progressive hint:** Labels are not automatically row positions.
   **Verify:** After reordering integer labels, assert `.loc` returns the requested labels and `.iloc` returns the requested positions; list both resulting indexes.
5. **Tracing:** Trace a boolean mask through creation, alignment by index, and row selection. What happens if mask/index labels differ?
   **Progressive hint:** Pandas aligns many labeled objects by index.
   **Verify:** Display mask and frame indexes side by side, then assert aligned selection for matching labels and the documented error/reindex policy for mismatches.
6. **Implementation:** Create a safe `tip_rate` column that yields missing values rather than infinity when `total_bill` is zero.
   **Progressive hint:** Mask or replace the zero denominator before division.
   **Verify:** Assert zero bills yield missing rates, positive bills yield the calculated ratio, and the entire column contains no positive/negative infinity.
7. **Debugging:** Repair chained assignment on a filtered DataFrame and explain when to use `.loc` or an explicit `.copy()`.
   **Progressive hint:** Make ownership and target rows explicit.
   **Verify:** Turn chained-assignment warnings into explicit `.loc` or `.copy()` ownership; assert the intended frame changes and the unintended frame does not.
8. **Edge case and explanation:** Compute a statistic for a possibly empty selection and return `None` instead of silently presenting `NaN` as a real result.
   **Progressive hint:** Check `.empty` before aggregating when absence has business meaning.
   **Verify:** Test nonempty and empty selections; assert the former returns the statistic and the latter returns `None`, not a value presented as meaningful.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Using the lesson DataFrame, select rows where `time == 'Dinner'`, then compute the mean of the selected `tip` values. **Before code:** state the row grain and write the mask separately. **Expected behavior:** the result is one scalar equal to `dinner_rows['tip'].mean()`. **Constraints:** use `.loc` and do not loop. **Verify:** assert every selected row is Dinner and the selection is non-empty before reporting the mean.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Using the lesson DataFrame, select rows where `time == 'Dinner'`, then compute the mean of the selected `tip` values. state the row grain and write the mask separately. the resu...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Add a Boolean `is_big_party` column that is `True` exactly when `size >= 5`. **Constraints:** use one vectorized comparison and explicit assignment; do not use row-wise `apply`. **Verify:** compare the new column with `df['size'].ge(5)`, inspect both True and False rows, and confirm row count/index are unchanged.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Add a Boolean `is_big_party` column that is `True` exactly when `size >= 5`. use one vectorized comparison and explicit assignment; do not use row-wise `apply`. compare the new...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** Create a safe `tip_rate = tip / total_bill`, treating a zero bill as missing rather than infinity, then sort descending by `tip_rate`. **Constraints:** preserve the unsorted source in a separate name and choose where missing rates appear. **Verify:** assert there are no infinite values and that non-missing rates are monotonically decreasing.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Create a safe `tip_rate = tip / total_bill`, treating a zero bill as missing rather than infinity, then sort descending by `tip_rate`. preserve the unsorted source in a separate...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict the difference between `.loc[labels]` and `.iloc[positions]` after an integer index has been reordered. **Progressive hint:** Labels are not automatically row positions. **Verify:** After reordering integer labels, assert `.loc` returns the requested labels and `.iloc` returns the requested positions; list both resulting indexes.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Predict the difference between `.loc[labels]` and `.iloc[positions]` after an integer index has been reordered. Labels are not automatically row positions. After reordering inte...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace a boolean mask through creation, alignment by index, and row selection. What happens if mask/index labels differ? **Progressive hint:** Pandas aligns many labeled objects by index. **Verify:** Display mask and frame indexes side by side, then assert aligned selection for matching labels and the documented error/reindex policy for mismatches.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Trace a boolean mask through creation, alignment by index, and row selection. What happens if mask/index labels differ? Pandas aligns many labeled objects by index. Display mask...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Create a safe `tip_rate` column that yields missing values rather than infinity when `total_bill` is zero. **Progressive hint:** Mask or replace the zero denominator before division. **Verify:** Assert zero bills yield missing rates, positive bills yield the calculated ratio, and the entire column contains no positive/negative infinity.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Create a safe `tip_rate` column that yields missing values rather than infinity when `total_bill` is zero. Mask or replace the zero denominator before division. Assert zero bill...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair chained assignment on a filtered DataFrame and explain when to use `.loc` or an explicit `.copy()`. **Progressive hint:** Make ownership and target rows explicit. **Verify:** Turn chained-assignment warnings into explicit `.loc` or `.copy()` ownership; assert the intended frame changes and the unintended frame does not.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Repair chained assignment on a filtered DataFrame and explain when to use `.loc` or an explicit `.copy()`. Make ownership and target rows explicit. Turn chained-assignment warni...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 8 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Compute a statistic for a possibly empty selection and return `None` instead of silently presenting `NaN` as a real result. **Progressive hint:** Check `.empty` before aggregating when absence has business meaning. **Verify:** Test nonempty and empty selections; assert the former returns the statistic and the latter returns `None`, not a value presented as meaningful.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 8 — your work
# Short contract: Compute a statistic for a possibly empty selection and return `None` instead of silently presenting `NaN` as a real result. Check `.empty` before aggregating when absence has bu...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
